In [ ]:
import requests
import time
from notebookutils import mssparkutils
from datetime import datetime, timezone, timedelta
from pyspark.sql.functions import max as spark_max
from pyspark.sql.types import StructType, StructField, TimestampType, StringType

execution_start_time = datetime.now(timezone.utc)

# --- Configuration ---
kusto_cluster_uri = "https://trd-6uegjpfbf030eemxtw.z1.kusto.fabric.microsoft.com"
kusto_database    = "MonitoringEventhouse"

tenant_id          = "your-tenant-id"
client_id          = "your-client-id"
client_secret      = "your-client-secret"
# client_secret    = mssparkutils.credentials.getSecret("your-kv", "your-secret")

dataverse_env_url  = "https://your-environment.crm4.dynamics.com"

# Single shared watermark config table (rows identified by TableName)
dataverse_config_table = "dataverse_load_config"

# Table 1: workspace owner (affirmations)
owner_staging_table = "workspace_owner_staging"
owner_target_table  = "workspace_owner"

# Table 2: workspace creation request
wcr_staging_table = "DataverseAOP_Staging"
wcr_target_table  = "DataverseAOP"


In [ ]:
# Create shared watermark config table in Lakehouse (if not exists)
# Each pipeline has its own row identified by TableName.
# On first run a row is absent — no watermark means all Dataverse records are fetched.
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {dataverse_config_table} (
        TableName        STRING,
        LastRunTimestamp TIMESTAMP,
        UpdatedAt        TIMESTAMP
    )
    USING DELTA
""")
print(f"Verified configuration table: {dataverse_config_table}")


In [ ]:
def get_spn_token(scope):
    url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    payload = {
        "grant_type":    "client_credentials",
        "client_id":     client_id,
        "client_secret": client_secret,
        "scope":         scope,
    }
    try:
        response = requests.post(url, data=payload)
        response.raise_for_status()
        return response.json().get("access_token")
    except Exception as e:
        print(f"Failed to retrieve token for scope {scope}: {e}")
        return None

def get_tokens():
    dv_scope  = dataverse_env_url.rstrip("/") + "/.default"
    dv_token  = get_spn_token(dv_scope)
    kql_token = get_spn_token(f"{kusto_cluster_uri}/.default")
    if not dv_token or not kql_token:
        raise Exception("ERROR: one or more tokens missing.")
    print("Tokens retrieved.")
    return dv_token, kql_token


In [ ]:
def get_watermark(table_name):
    df_config = spark.table(dataverse_config_table).filter(f"TableName = '{table_name}'")
    max_row   = df_config.agg(spark_max("LastRunTimestamp").alias("max_ts")).collect()
    max_ts    = max_row[0]["max_ts"]
    if max_ts:
        ts = max_ts
        if ts.tzinfo is None:
            ts = ts.replace(tzinfo=timezone.utc)
        print(f"[{table_name}] Watermark loaded: {ts}")
        return ts
    print(f"[{table_name}] No watermark found — will fetch all records.")
    return None

def update_watermark(table_name):
    upper_bound_str = execution_start_time.strftime("%Y-%m-%dT%H:%M:%S")
    try:
        row_count = spark.table(dataverse_config_table).filter(f"TableName = '{table_name}'").count()
        if row_count == 0:
            spark.sql(f"""
                INSERT INTO {dataverse_config_table}
                VALUES ('{table_name}', CAST('{upper_bound_str}' AS TIMESTAMP), current_timestamp())
            """)
            print(f"[{table_name}] Watermark inserted (first run): {upper_bound_str}")
        else:
            spark.sql(f"""
                UPDATE {dataverse_config_table}
                SET LastRunTimestamp = CAST('{upper_bound_str}' AS TIMESTAMP),
                    UpdatedAt        = current_timestamp()
                WHERE TableName = '{table_name}'
            """)
            print(f"[{table_name}] Watermark updated to: {upper_bound_str}")
    except Exception as e:
        print(f"[{table_name}] WARNING: Failed to update watermark: {e}")


In [ ]:
def run_kusto_command(query, max_retries=5):
    """
    Executes a KQL control command via REST API.
    Retries on 429 TooManyRequests with exponential backoff.
    """
    if not kusto_token:
        print("Cannot run command: no Kusto token available.")
        return False

    mgmt_endpoint = f"{kusto_cluster_uri.rstrip('/')}/v1/rest/mgmt"
    headers = {
        "Authorization": f"Bearer {kusto_token}",
        "Content-Type":  "application/json",
    }
    body = {"db": kusto_database, "csl": query}

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(mgmt_endpoint, headers=headers, json=body)

            if response.status_code == 429:
                retry_after = int(response.headers.get("Retry-After", 0))
                wait_seconds = retry_after if retry_after > 0 else (2 ** attempt)
                print(f"429 throttled (attempt {attempt}/{max_retries}). Retrying in {wait_seconds}s...")
                time.sleep(wait_seconds)
                continue

            response.raise_for_status()
            return True

        except requests.exceptions.HTTPError as e:
            if attempt < max_retries:
                wait_seconds = 2 ** attempt
                print(f"HTTP error on attempt {attempt}/{max_retries}: {e}. Retrying in {wait_seconds}s...")
                time.sleep(wait_seconds)
            else:
                print(f"Failed after {max_retries} attempts: {e}")
                if response.content:
                    print(f"Details: {response.content}")
                return False
        except Exception as e:
            print(f"Unexpected error: {e}")
            return False

    print(f"Exhausted all {max_retries} retry attempts.")
    return False


In [ ]:
def fetch_dataverse_records(entity, columns, watermark_ts, dv_token, watermark_field="modifiedon"):
    api_version   = "v9.2"
    filter_clause = ""
    if watermark_ts:
        watermark_str = watermark_ts.strftime("%Y-%m-%dT%H:%M:%SZ")
        filter_clause = f"&$filter={watermark_field} gt {watermark_str}"

    url = f"{dataverse_env_url.rstrip('/')}/api/data/{api_version}/{entity}?{columns}{filter_clause}"

    headers = {
        "Authorization":    f"Bearer {dv_token}",
        "Content-Type":     "application/json",
        "OData-MaxVersion": "4.0",
        "OData-Version":    "4.0",
        "Prefer":           "odata.include-annotations=OData.Community.Display.V1.FormattedValue",
    }

    records = []
    while url:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        data = response.json()
        records.extend(data.get("value", []))
        url = data.get("@odata.nextLink")

    print(f"[{entity}] Fetched {len(records)} record(s) from Dataverse.")
    return records


In [ ]:
def write_to_staging(records, kql_token, staging_table, schema, row_mapper):
    if not records:
        print(f"[{staging_table}] No records to write — skipping staging write.")
        return

    rows       = [row_mapper(r) for r in records]
    df_staging = spark.createDataFrame(rows, schema)

    df_staging.write \
        .format("com.microsoft.kusto.spark.synapse.datasource") \
        .option("kustoCluster",       kusto_cluster_uri) \
        .option("kustoDatabase",      kusto_database) \
        .option("kustoTable",         staging_table) \
        .option("accessToken",        kql_token) \
        .option("tableCreateOptions", "CreateIfNotExist") \
        .mode("Append") \
        .save()

    print(f"[{staging_table}] Written {len(rows)} row(s).")


In [ ]:
def merge_to_target(records, merge_query, staging_table):
    if not records:
        print(f"[{staging_table}] No records staged — skipping merge.")
        return

    if not run_kusto_command(merge_query):
        raise Exception(f"Merge from '{staging_table}' failed.")
    print(f"[{staging_table}] Merge to target complete.")

    if run_kusto_command(f".clear table {staging_table} data"):
        print(f"[{staging_table}] Staging cleared.")
    else:
        print(f"[{staging_table}] WARNING: failed to clear staging table.")


In [ ]:
# --- Main execution ---
dataverse_token, kusto_token = get_tokens()

# ── Pipeline 1: workspace_owner (affirmations) ──────────────────────────────
owner_watermark = get_watermark(owner_target_table)

owner_records = fetch_dataverse_records(
    entity       = "ubsppcoe_workspaceaffirmations",
    columns      = "$select=modifiedon,_ubsppcoe_affirmedobjectid_value,ubsppcoe_primaryowneremail,ubsppcoe_secondaryowneremail",
    watermark_ts = owner_watermark,
    dv_token     = dataverse_token,
)

owner_schema = StructType([
    StructField("workspaceId",    StringType(), True),
    StructField("PrimaryEmail",   StringType(), True),
    StructField("SecondaryEmail", StringType(), True),
    StructField("modifiedon",     StringType(), True),
])
owner_row_mapper = lambda r: (
    r.get("_ubsppcoe_affirmedobjectid_value"),
    r.get("ubsppcoe_primaryowneremail"),
    r.get("ubsppcoe_secondaryowneremail"),
    r.get("modifiedon"),
)

owner_merge_query = f"""
.set-or-append {owner_target_table} <|
let staged = {owner_staging_table}
    | where isnotnull(workspaceId)
    | summarize arg_max(modifiedon, *) by workspaceId
    | project workspaceId,
              PrimaryEmail   = coalesce(PrimaryEmail, ""),
              SecondaryEmail = coalesce(SecondaryEmail, ""),
              modifiedon;
let existing = {owner_target_table}
    | summarize arg_max(modifiedon, *) by workspaceId
    | project workspaceId,
              PrimaryEmail   = coalesce(PrimaryEmail, ""),
              SecondaryEmail = coalesce(SecondaryEmail, "");
staged
| join kind=leftanti existing on workspaceId, PrimaryEmail, SecondaryEmail
| project workspaceId,
          PrimaryEmail   = iif(PrimaryEmail == "",   tostring(dynamic(null)), PrimaryEmail),
          SecondaryEmail = iif(SecondaryEmail == "", tostring(dynamic(null)), SecondaryEmail),
          modifiedon
"""

write_to_staging(owner_records, kusto_token, owner_staging_table, owner_schema, owner_row_mapper)
merge_to_target(owner_records, owner_merge_query, owner_staging_table)
update_watermark(owner_target_table)

# ── Pipeline 2: WorkspaceCreationRequest ────────────────────────────────────
wcr_watermark = get_watermark(wcr_target_table)

wcr_records = fetch_dataverse_records(
    entity       = "ubsppcoe_workspacecreationrequests",
    columns      = "$select=ubsppcoe_workspaceid,ubsppcoe_oapenabled,createdon,modifiedon",
    watermark_ts = wcr_watermark,
    dv_token     = dataverse_token,
)

wcr_schema = StructType([
    StructField("workspaceId", StringType(), True),
    StructField("AOPEnabled",  StringType(), True),
    StructField("createdon",   StringType(), True),
    StructField("modifiedon",  StringType(), True),
])
wcr_row_mapper = lambda r: (
    r.get("ubsppcoe_workspaceid"),
    str(r.get("ubsppcoe_oapenabled")) if r.get("ubsppcoe_oapenabled") is not None else None,
    r.get("createdon"),
    r.get("modifiedon"),
)

wcr_merge_query = f"""
.set-or-append {wcr_target_table} <|
let staged = {wcr_staging_table}
    | where isnotnull(WorkspaceId)
    | summarize arg_max(modifiedon, *) by WorkspaceId
    | project WorkspaceId,
              Activity   = iif(coalesce(tobool(AOPEnabled), false), "Enabled", "Disabled"),
              createdon  = todatetime(createdon),
              modifiedon = todatetime(modifiedon);
let existing = {wcr_target_table}
    | summarize arg_max(modifiedon, *) by WorkspaceId
    | project WorkspaceId, Activity, createdon;
staged
| join kind=leftanti existing on WorkspaceId, Activity, createdon
| project WorkspaceId, Activity, createdon, modifiedon
"""

write_to_staging(wcr_records, kusto_token, wcr_staging_table, wcr_schema, wcr_row_mapper)
merge_to_target(wcr_records, wcr_merge_query, wcr_staging_table)
update_watermark(wcr_target_table)
